# 🧠 Workshop: Building Blocks for AI Agents

## NLP Pipeline + Probabilistic Language Models (90-Minute Team Lab)

**Objective:**
Work in teams of 3 to build a small NLP pipeline and implement unigram and bigram models, culminating in estimating sentence probabilities. Submit your completed Jupyter Notebook via a GitHub link (with `.git` at the end).

## Part 1 – NLP Pipeline

### Step 1: Select and Load a Corpus

Select a corpus from `nltk`, or upload your own text documents. Ensure your vocabulary size exceeds 2000 words.

In [10]:
import nltk
nltk.download('reuters')
from nltk.corpus import reuters

corpus_docs = [reuters.raw(fileid) for fileid in reuters.fileids()]
corpus_text = " ".join(corpus_docs)

[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\85155\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!


**👨‍🏫 Professor Talking Point:** This corpus is pre-tokenized and covers multiple topics. It’s a good fit to get us above the 2,000-word vocabulary requirement.

### Step 2: Collect and Preprocess Documents

Convert your corpus into tokens and compute the vocabulary size.

In [11]:
from nltk.tokenize import word_tokenize

nltk.download('punkt')
tokens = word_tokenize(corpus_text.lower())
vocab = set(tokens)
print(f"Vocabulary size: {len(vocab)}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\85155\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Vocabulary size: 52440


In [12]:
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Ensure 'punkt' is available and nltk_data path is set
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', download_dir=nltk_data_path, force=True)
print("Downloading 'punkt_tab' tokenizer...")
nltk.download('punkt_tab', download_dir=nltk_data_path, force=True)

# Always append the custom nltk_data path (if not already present)
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

# Debugging paths and contents
print("NLTK Data Paths:", nltk.data.path)
print("Contents of nltk_data:", os.listdir(nltk_data_path))

tokens = word_tokenize(corpus_text.lower())
vocab = set(tokens)
print(f"Vocabulary size: {len(vocab)}")

[nltk_data] Downloading package punkt to d:\Study\Project\NLP_Probabil
[nltk_data]     isticLanguageModels\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to d:\Study\Project\NLP_Prob
[nltk_data]     abilisticLanguageModels\nltk_data...


[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


NLTK Data Paths: ['C:\\Users\\85155/nltk_data', 'd:\\Study\\Project\\NLP_ProbabilisticLanguageModels\\venv\\nltk_data', 'd:\\Study\\Project\\NLP_ProbabilisticLanguageModels\\venv\\share\\nltk_data', 'd:\\Study\\Project\\NLP_ProbabilisticLanguageModels\\venv\\lib\\nltk_data', 'C:\\Users\\85155\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'd:\\Study\\Project\\NLP_ProbabilisticLanguageModels\\nltk_data']
Contents of nltk_data: ['tokenizers']
Vocabulary size: 52440


**👨‍🏫 Professor Talking Point:** Vocabulary size is important—it determines the richness of our model. Models trained on small vocabularies can't generalize well.

### Step 3: Implement Tokenizer

In [13]:
import re

def simple_tokenizer(text):
    return re.findall(r'\b\w+\b', text.lower())

tokens = simple_tokenizer(corpus_text)

**👨‍🏫 Professor Talking Point:** A simple regex tokenizer gives us control—this is useful when we need to understand every processing step.

### Step 4: Normalization, Stemming, and Stopword Removal

In [14]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

nltk.download('stopwords')

def normalize(tokens):
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()
    return [stemmer.stem(word) for word in tokens if word not in stop_words and word not in string.punctuation]

normalized_tokens = normalize(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\85155\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**👨‍🏫 Professor Talking Point:** Normalization makes the data more consistent and shrinks the vocabulary. This is essential for estimating reliable probabilities.

### Step 5: Build an Inverted Index

An **inverted index** maps each token to the list of positions where it appears in the corpus.  
This is a foundational NLP data structure used in search engines and information retrieval systems.

It helps us:
- quickly locate words in the corpus,
- support search and retrieval tasks,
- understand how token-level indexing works before moving into probabilistic language models.

In [15]:
from collections import defaultdict

inverted_index = defaultdict(list)

for i, token in enumerate(normalized_tokens):
    inverted_index[token].append(i)

print(f"Number of indexed terms: {len(inverted_index)}")
print("Sample entries:")
for term in list(inverted_index.keys())[:5]:
    print(term, "->", inverted_index[term][:10])

Number of indexed terms: 23287
Sample entries:
asian -> [0, 32, 108489, 109104, 115752, 120989, 121004, 128243, 154734, 157326]
export -> [1, 17, 47, 101, 131, 219, 244, 341, 348, 389]
fear -> [2, 13, 11736, 11783, 11830, 12164, 24543, 38382, 42894, 67019]
damag -> [3, 25, 5453, 7528, 7759, 9471, 11791, 12167, 24069, 33513]
u -> [4, 10, 34, 41, 60, 135, 155, 180, 206, 212]


## Part 2 – Probabilistic Language Models

### 📘 Unigram Model

A **Unigram Model** is a type of probabilistic language model that assumes each word in a sentence is **independent** of the words that came before it.

The probability of a sequence of words $w_1, w_2, ..., w_n$ is calculated as:

$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$

To estimate $P(w_i)$, we use the **Maximum Likelihood Estimate (MLE)**:

$$
P(w_i) = \frac{\text{count}(w_i)}{\sum_{j} \text{count}(w_j)}
$$

where $j$ is the total number of words in the corpus.

This is a strong simplification, but it provides a foundational baseline and helps reduce data sparsity in low-resource environments.

Here's how to implement it:


In [16]:
from collections import Counter

unigram_counts = Counter(normalized_tokens)
total_words = len(normalized_tokens)

def unigram_prob(word):
    return unigram_counts[word] / total_words

print(f"P('japan') = {unigram_prob('japan'):.5f}")
print(f"P('citibank') = {unigram_prob('citibank'):.5f}")
print(f"P('harvest') = {unigram_prob('harvest'):.5f}")
print(f"P('bank') = {unigram_prob('bank'):.5f}")


P('japan') = 0.00185
P('citibank') = 0.00005
P('harvest') = 0.00024
P('bank') = 0.00493


##### 📘 Why Are Unigram Probabilities So Low?

Unigram probabilities represent the **relative frequency** of individual words in the entire corpus:

$$
P(w_i) = \frac{\text{count}(w_i)}{\text{total number of tokens in the corpus}}
$$

In our case, the total number of tokens is quite large:

- **Total tokens:** 1,178,604  
- **Unique words (vocabulary size):** 67,151

Even if a word appears frequently, its probability will still be small relative to the total number of tokens.

For example:
- `"bank"` appears quite often, yet its probability is only **0.00493**, or about **0.5%** of the total words.
- `"citibank"` appears only a few times, resulting in a much smaller probability of **0.00005**.

These small values are expected when:
- The corpus is **large and diverse** (like Reuters).
- Many words appear **only once or twice**, which is common in natural language (known as Zipf's Law).

**Conclusion:**  
Low unigram probabilities do **not** indicate an error—they reflect a realistic distribution of word frequencies across a large corpus. This also highlights the need for smoothing when building more complex language models.


### 📘 Chain Rule with Unigrams

Using the **Chain Rule**, we estimate the probability of a sequence:
$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$
This is a simplifying assumption of complete independence (unrealistic but foundational).

**👨‍🏫 Professor Talking Point:** Unigram models assume word independence—useful but limited since word order is ignored.

In [17]:
def sentence_prob_unigram(sentence):
    words = normalize(simple_tokenizer(sentence))
    prob = 1.0
    for word in words:
        prob *= unigram_prob(word)
    return prob

sentence = "The agreement calls for joint Sino-Chilean management of the venture for 15 years, the paper said"
print(sentence_prob_unigram(sentence))

2.382179640797073e-37


##### 📘 Why Is the Sentence Probability So Low?

The calculated **unigram sentence probability** is:

```python
2.382179640797073e-37
````

This number is extremely small—but **that’s expected** for long sentences under a unigram model. Here's why:


##### 🔢 Corpus Statistics

* **Total number of tokens:** 1,178,604
* **Vocabulary size (unique tokens):** 67,151

##### 📉 How the Unigram Model Works

The unigram model computes sentence probability as the **product of individual word probabilities**:

$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$

Each word typically has a probability between 0.00001 and 0.01. When multiplying **10–20 small numbers together**, the final result becomes **exponentially smaller**, approaching zero for longer sentences.

##### 🧪 Impact of Preprocessing (Step 4)

The normalization step involves:

* Lowercasing
* **Stop word removal** (e.g., "the", "of", "for", "said")
* **Stemming** (e.g., "management" → "manag")
* **Punctuation removal**

This reduces the number of words used in the calculation. While this makes the vocabulary smaller and more manageable, it also means:

* **Common but removed words** (like "the") don’t contribute to the probability.
* **Stemmed forms** may not match original unigrams perfectly (e.g., “sino-chilean” becomes `sinochilean` or `sino` and `chilean`, depending on the tokenizer).

So even though the sentence appears long, **only 7–12 stemmed and filtered tokens** may remain after preprocessing—yet each one still has a very small individual probability.

##### ✅ Key Takeaways

* Low sentence probabilities are **normal** in unigram models, especially for longer sentences.
* The **multiplicative nature** of probability and the **sparsity of natural language** lead to very small final values.
* These limitations are one reason why more advanced models (like bigrams or neural LMs) are needed for realistic NLP applications.

You can inspect the intermediate tokens like this:

```python
print(normalize(simple_tokenizer(sentence)))
```


### **💭 Thinking Point**

**What is the limitation of the unigram model?**  
I think the biggest limitation is that it ignores word order.  
It only looks at individual word frequency, so the sentence structure is not considered.  
Because of this, the result may not reflect real language very well.

### 📘 Smoothed Unigram Model (Add-One / Laplace Smoothing)

A problem with the basic unigram model is that any unseen word gets probability **0**.  
To reduce this issue, we apply **Laplace smoothing**:

$$
P(w) = \frac{\text{count}(w) + 1}{N + V}
$$

where:

- $N$ = total number of tokens
- $V$ = vocabulary size

This ensures every word receives a non-zero probability.

In [18]:
V = len(unigram_counts)

def unigram_prob_smoothed(word):
    return (unigram_counts[word] + 1) / (total_words + V)

print(f"Smoothed P('japan') = {unigram_prob_smoothed('japan'):.8f}")
print(f"Smoothed P('citibank') = {unigram_prob_smoothed('citibank'):.8f}")
print(f"Smoothed P('unknownword') = {unigram_prob_smoothed('unknownword'):.8f}")

Smoothed P('japan') = 0.00181040
Smoothed P('citibank') = 0.00004500
Smoothed P('unknownword') = 0.00000096


In [19]:
def sentence_prob_unigram_smoothed(sentence):
    words = normalize(simple_tokenizer(sentence))
    prob = 1.0
    for word in words:
        prob *= unigram_prob_smoothed(word)
    return prob

sentence = "The agreement calls for joint Sino-Chilean management of the venture for 15 years, the paper said"
print("Smoothed unigram sentence probability:", sentence_prob_unigram_smoothed(sentence))

Smoothed unigram sentence probability: 2.7115595810420195e-37


### **💭 Thinking Point**

**Why do we apply smoothing to the unigram model?**  
Smoothing is used to avoid zero probability for unseen words.  
Without smoothing, one unknown word can make the whole sentence probability zero.  
So smoothing makes the model more stable.

### 📘 Bigram Model with MLE – Mathematical Explanation

The **Bigram Model** assumes the current word depends only on the previous word.
The MLE (Maximum Likelihood Estimate) for a bigram $(w_{i-1}, w_i)$ is:
$$
P(w_i | w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}
$$

**👨‍🏫 Professor Talking Point:** This simple multiplication illustrates the chain rule, but we’ll soon see how to improve this with context.

### 📘 Sentence Probability with Bigram Model – Mathematical Explanation

Using the bigram model and chain rule:
$$
P(w_1, w_2, ..., w_n) = P(w_1) \cdot P(w_2 | w_1) \cdot P(w_3 | w_2) \cdots P(w_n | w_{n-1})
$$
This models **local dependencies** between words.

In [20]:
from collections import defaultdict

bigram_counts = defaultdict(int)

for i in range(len(normalized_tokens) - 1):
    pair = (normalized_tokens[i], normalized_tokens[i + 1])
    bigram_counts[pair] += 1

def bigram_prob(w1, w2):
    return bigram_counts[(w1, w2)] / unigram_counts[w1] if unigram_counts[w1] > 0 else 0

**👨‍🏫 Professor Talking Point:** Bigram probabilities model word context, capturing more meaning than unigrams.

### Sentence Probability with Bigram Model

In [21]:
def sentence_prob_bigram(sentence):
    words = normalize(simple_tokenizer(sentence))
    prob = 1.0
    for i in range(len(words) - 1):
        prob *= bigram_prob(words[i], words[i + 1])
    return prob

sentence = "The agreement calls for joint Sino-Chilean management of the venture for 15 years, the paper said"
print(sentence_prob_bigram(sentence))

5.6991255325977075e-21


**👨‍🏫 Professor Talking Point:** Estimating sentence probability using bigrams shows how sequence information improves prediction power.

### **💭 Thinking Point**

**Why is the bigram model more realistic than unigram?**  
Bigram model is more realistic because it considers the previous word.  
This helps capture local context in a sentence.  
However, it still has problem when some word pairs never appear in the dataset.

### 📘 Smoothed Bigram Model (Add-One / Laplace Smoothing)

Basic bigram probabilities often become zero when a word pair never appeared in training.  
To handle this, we use **Laplace smoothing**:

$$
P(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i) + 1}{\text{count}(w_{i-1}) + V}
$$

where $V$ is the vocabulary size.

In [22]:
def bigram_prob_smoothed(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

In [23]:
def sentence_prob_bigram_smoothed(sentence):
    words = normalize(simple_tokenizer(sentence))
    if len(words) < 2:
        return unigram_prob_smoothed(words[0]) if words else 0
    
    prob = unigram_prob_smoothed(words[0])
    for i in range(len(words) - 1):
        prob *= bigram_prob_smoothed(words[i], words[i + 1])
    return prob

sentence = "The agreement calls for joint Sino-Chilean management of the venture for 15 years, the paper said"
print("Smoothed bigram sentence probability:", sentence_prob_bigram_smoothed(sentence))

Smoothed bigram sentence probability: 1.1499420994665575e-40


### **💭 Thinking Point**

**What is the advantage of smoothing in bigram models?**  
In bigram model, many word pairs may not appear in training data.  
Smoothing helps assign small probability to these unseen pairs.  
I think this makes the model more useful in real situations.

### 📊 Model Comparison

Now we compare the sentence probabilities produced by:
1. Unigram MLE
2. Unigram with Laplace smoothing
3. Bigram MLE
4. Bigram with Laplace smoothing

This helps us observe how context and smoothing affect probability estimation.

In [24]:
sentence = "The agreement calls for joint Sino-Chilean management of the venture for 15 years, the paper said"

results = {
    "Unigram MLE": sentence_prob_unigram(sentence),
    "Unigram Smoothed": sentence_prob_unigram_smoothed(sentence),
    "Bigram MLE": sentence_prob_bigram(sentence),
    "Bigram Smoothed": sentence_prob_bigram_smoothed(sentence)
}

for model, prob in results.items():
    print(f"{model}: {prob:.5e}")

Unigram MLE: 2.38218e-37
Unigram Smoothed: 2.71156e-37
Bigram MLE: 5.69913e-21
Bigram Smoothed: 1.14994e-40


### ✅ Observation

From the comparison, we can see:

- **Unigram models** ignore word order, so they are simpler but less realistic.
- **Bigram models** consider local context, so they capture sentence structure better.
- **Smoothed models** avoid zero probabilities and are more robust for unseen words or word pairs.
- In this experiment, **Bigram MLE produces the highest probability**, while smoothing reduces the probability but improves robustness against unseen data.

### **💭 Thinking Point**

**When should we use unigram, bigram, or smoothed models?**  
I think unigram model is useful when we only care about general word frequency and want a simple model.  
Bigram model is better when we need to consider word order and local context.  
If the dataset is small or has many unseen words, smoothed models are more suitable because they avoid zero probability.

## Part 3: The Workshop


One team member must push the final notebook to GitHub and send the `.git` URL to the instructor before the end of class.

## 🧠 Learning Objectives
- Implement the foundations of **Probabilistic Language Models** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(20 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(65 min)* – NLP Pipeline and four Probabilistic Language Model method implementations + Markdown documentation (work as teams)
3. **Push to GitHub** *(5 min)* – Teams commit and push the one notebook. **Make sure to include your names so it is easy to identify the team that developed the code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(1 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Probabilistic Language Models Workshop, Team #_____.


## 💻 Submission Checklist
- ✅ `ProbabilisticLanguageModels.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, Inverted Index and the four methods.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** (1-2 per concept)
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `ProbabilisticLanguageModels`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 💭 Thinking Points

### 1. Unigram Model
**Q: What is the limitation of the unigram model?**  
A: I think the unigram model is too simple because it ignores word order.  
It only uses word frequency, so it cannot represent sentence structure well.

---

### 2. Bigram Model
**Q: Why is the bigram model more realistic than unigram?**  
A: Bigram model considers the previous word, so it captures local context.  
This makes the sentence more natural, but it still has problem when some word pairs do not appear.

---

### 3. Smoothed Models
**Q: Why is smoothing important in language models?**  
A: Smoothing is important because it avoids zero probability for unseen words or pairs.  
Without smoothing, one unknown word can make the whole sentence probability zero.

---

### 4. Model Selection
**Q: When should we use different language models?**  
A: Unigram model is useful for simple tasks and fast computation.  
Bigram model is better when we need context.  
In real-world data, I think smoothed models are more practical because data is often incomplete.

## 🧭 Conclusion

Today you’ve constructed your own basic language model. Next class, we’ll expand these ideas to explore **Large Language Models (LLMs)**—like ChatGPT—which learn patterns over **massive corpora** using **deep neural networks** instead of just counts.